# NB-R12 — Regenerate Figures from Corrected Results

**Pipeline stage:** 12 of 13

**Purpose.** Regenerate figures that were found, during an internal audit, to still display pre-correction numbers even though the corresponding tables and text had already been updated earlier in the pipeline -- e.g. a regime-accuracy figure still showing the original biased 84.5%/54.8% split with significance asterisks, an equity-curve figure still labelled with the original, look-ahead-biased VIX threshold (15.08) and shading multiple trades where the corrected protocol produces exactly one, and a VIX-vs-GARCH comparison figure still showing superseded sample sizes (n=58/n=219). This notebook rebuilds each of those figures directly from the corrected result files so every number displayed is traceable to an actual computation.

**Dependency on NB-R13:** Sections 1 and 2 (regime accuracy and AUC by horizon) load their 1-day/5-day values directly from `results/horizon_1d_5d_results_CORRECTED.csv` and `data/processed/test_predictions_{1d,5d}.csv`, both produced by NB-R13. Run NB-R13 before this notebook if those files do not yet exist.

**Inputs:** `data/processed/test_predictions.csv`, `test_predictions_1d.csv`, `test_predictions_5d.csv`, `data/raw/market_data.csv`, `results/statistical_tests.json`, `results/table4_regime_metrics.csv`, `results/horizon_1d_5d_results_CORRECTED.csv` (from NB-R13), `results/table10_garch_comparison_CORRECTED.csv` (from NB-R11).

**Outputs:** `plots/R12_Fig3_regime_accuracy_by_horizon.png`, `plots/R12_Fig4_auc_by_horizon.png`, `plots/R12_Fig5_equity_curves.png`, `plots/R12_Fig6_drawdown.png`, `plots/R12_Fig8_vix_vs_garch.png`.

**Note:** two other figures (transaction-cost sensitivity and bootstrap/permutation diagnostics) already had correct, up-to-date versions available from NB-R09 and NB-R04 respectively (`plots/R09_trading_simulation.png`, `plots/R04_statistical_tests.png`) and are reused as-is rather than regenerated here.


## Setup

In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
from pathlib import Path

PROJ = Path('..').resolve()  # repo root, assuming this notebook is run from notebooks/
PROC = PROJ / 'data' / 'processed'
RAW = PROJ / 'data' / 'raw'
RESULTS = PROJ / 'results'
PLOTS = PROJ / 'plots'

RFR_ANNUAL, TRADING_DAYS = 0.065, 252
RFR_DAILY = RFR_ANNUAL / TRADING_DAYS


## 1. Regime-Conditioned Accuracy by Horizon

Loads real 1-day, 5-day (NB-R13), and 21-day (NB-R04) regime-conditioned results -- no hardcoded placeholders.

In [ ]:
# Load real regime-conditioned results for all three horizons.
# 1-day/5-day come from NB-R13 (actually retrained on the corrected split);
# 21-day comes from NB-R04's regime-metrics table.
h1d5d = pd.read_csv(RESULTS / 'horizon_1d_5d_results_CORRECTED.csv').set_index('horizon')
t4 = pd.read_csv(RESULTS / 'table4_regime_metrics.csv')  # columns: Metric, High-VIX, Low-VIX (21d)
acc21_hv = float(t4.loc[t4['Metric'] == 'Accuracy (%)', 'High-VIX'].iloc[0].strip('%')) / 100
acc21_lv = float(t4.loc[t4['Metric'] == 'Accuracy (%)', 'Low-VIX'].iloc[0].strip('%')) / 100
n21_hv = int(t4.loc[t4['Metric'] == 'N', 'High-VIX'].iloc[0])
n21_lv = int(t4.loc[t4['Metric'] == 'N', 'Low-VIX'].iloc[0])

tbl7 = pd.DataFrame([
    dict(horizon='1d', hv_n=int(h1d5d.loc['1d', 'hv_n']), hv_acc=h1d5d.loc['1d', 'hv_acc'],
         lv_n=int(h1d5d.loc['1d', 'lv_n']), lv_acc=h1d5d.loc['1d', 'lv_acc'], fisher_p=h1d5d.loc['1d', 'fisher_p']),
    dict(horizon='5d', hv_n=int(h1d5d.loc['5d', 'hv_n']), hv_acc=h1d5d.loc['5d', 'hv_acc'],
         lv_n=int(h1d5d.loc['5d', 'lv_n']), lv_acc=h1d5d.loc['5d', 'lv_acc'], fisher_p=h1d5d.loc['5d', 'fisher_p']),
    dict(horizon='21d', hv_n=n21_hv, hv_acc=acc21_hv, lv_n=n21_lv, lv_acc=acc21_lv, fisher_p=1.000),
])
print(tbl7)

fig, ax = plt.subplots(figsize=(7.5, 4.8))
x = np.arange(len(tbl7))
w = 0.35
ax.bar(x - w/2, tbl7['hv_acc']*100, w, label='High-VIX (n=8)', color='#D6604D')
ax.bar(x + w/2, tbl7['lv_acc']*100, w, label='Low-VIX (n=248)', color='#4393C3')
for i, row in tbl7.iterrows():
    ax.text(i - w/2, row['hv_acc']*100 + 1.5, f"{row['hv_acc']*100:.1f}%", ha='center', fontsize=8)
    ax.text(i + w/2, row['lv_acc']*100 + 1.5, f"{row['lv_acc']*100:.1f}%", ha='center', fontsize=8)
ax.axhline(50, color='grey', linestyle=':', linewidth=1, label='Coin flip')
ax.set_xticks(x)
xlabels = [
    "1-day\np={:.3f}".format(tbl7.loc[0, 'fisher_p']),
    "5-day\np={:.3f} (naive)".format(tbl7.loc[1, 'fisher_p']),
    "21-day\np=1.000 (non-overlap)",
]
ax.set_xticklabels(xlabels)
ax.set_ylabel('Directional Accuracy (%)')
ax.set_title('Regime-Conditioned Directional Accuracy by Horizon (real retrained 1d/5d models)\n'
             'Same 8 High-VIX days across all horizons; only 21-day has independent dependence-aware validation')
ax.set_ylim(0, 115)
ax.legend(fontsize=8, loc='upper left')
plt.tight_layout()
plt.savefig(PLOTS / 'R12_Fig3_regime_accuracy_by_horizon.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved Fig 3')


## 2. AUC by Horizon

AUC is computed directly per horizon and left undefined (not plotted as a fabricated number) wherever the underlying subset is single-class -- e.g. the High-VIX subset at the 5-day and 21-day horizons, where all 8 observations realize the same outcome.

In [ ]:
from sklearn.metrics import roc_auc_score

# Unconditioned (overall) AUC per horizon, from NB-R13 output + statistical_tests.json for 21d.
with open(RESULTS / 'statistical_tests.json') as f:
    stats21 = json.load(f)
auc_uncond = {
    '1-day': float(h1d5d.loc['1d', 'overall_auc']),
    '5-day': float(h1d5d.loc['5d', 'overall_auc']),
    '21-day': float(stats21['stack_overall_auc']),
}

# Regime-conditioned AUC, computed directly per horizon. Left as NaN wherever the
# High-VIX subset is single-class (AUC undefined), rather than hardcoded.
def regime_auc(path, target_col):
    df = pd.read_csv(path)
    hv = df[df['regime_fixed'] == 'High-VIX']
    lv = df[df['regime_fixed'] != 'High-VIX']
    hv_auc = roc_auc_score(hv[target_col], hv['stack_prob']) if hv[target_col].nunique() > 1 else np.nan
    lv_auc = roc_auc_score(lv[target_col], lv['stack_prob']) if lv[target_col].nunique() > 1 else np.nan
    return hv_auc, lv_auc

auc_hv_1d, auc_lv_1d = regime_auc(PROC / 'test_predictions_1d.csv', 'dir_1d')
auc_hv_5d, auc_lv_5d = regime_auc(PROC / 'test_predictions_5d.csv', 'dir_5d')
auc_lowvix_21d = float(stats21['low_vix']['auc'])
auc_highvix_21d = np.nan  # single class (n=8, all Up) -- undefined by construction

auc_lowvix = {'1-day': auc_lv_1d, '5-day': auc_lv_5d, '21-day': auc_lowvix_21d}
auc_highvix = {'1-day': auc_hv_1d, '5-day': auc_hv_5d, '21-day': auc_highvix_21d}
print('High-VIX AUC by horizon:', auc_highvix)
print('Low-VIX AUC by horizon:', auc_lowvix)

fig, ax = plt.subplots(figsize=(7.5, 4.2))
horizons = ['1-day', '5-day', '21-day']
xw = np.arange(len(horizons))
ax.plot(xw, [auc_uncond[h] for h in horizons], 'o-', label='Unconditioned (overall)', color='grey')
ax.plot(xw, [auc_lowvix[h] for h in horizons], 's-', label='Low-VIX', color='#4393C3')
hv_computable = [(i, auc_highvix[h]) for i, h in enumerate(horizons) if not np.isnan(auc_highvix[h])]
if hv_computable:
    xs, ys = zip(*hv_computable)
    ax.plot(xs, ys, '^', color='#D6604D', markersize=10, label='High-VIX (only where computable)')
ax.axhline(0.5, color='black', linestyle=':', linewidth=1, label='AUC = 0.5 (no ranking skill)')
ax.set_xticks(xw)
ax.set_xticklabels(horizons)
ax.set_ylabel('AUC')
ax.set_ylim(0.40, 0.65)
ax.set_title('Threshold-Free Ranking Quality (AUC) by Horizon\n'
             'High-VIX AUC undefined wherever the subset is single-class (n=8, all Up)')
ax.legend(fontsize=7.5, loc='lower right')
for i, h in enumerate(horizons):
    if np.isnan(auc_highvix[h]):
        ax.text(i, 0.42, 'High-VIX\nAUC: N/A\n(single class)', ha='center', fontsize=7.5, color='#D6604D')
plt.tight_layout()
plt.savefig(PLOTS / 'R12_Fig4_auc_by_horizon.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved Fig 4')


## 3. Equity Curves and Drawdown

Reconstruct the day-by-day strategy and buy-and-hold equity curves under the corrected execution protocol (signal at close[t], entry at open[t+1], exit at close[t+21], single position only), exactly reproducing the summary statistics reported in NB-R09.

In [ ]:
test = pd.read_csv(PROC / 'test_predictions.csv', parse_dates=['date'])
mkt = pd.read_csv(RAW / 'market_data.csv', parse_dates=['date'])
mkt.columns = [c.strip().lower().replace(' ', '_') for c in mkt.columns]
close_col = [c for c in mkt.columns if 'close' in c and 'india' not in c][0]
open_col = [c for c in mkt.columns if 'open' in c][0]
mkt = mkt[['date', close_col, open_col]].rename(columns={close_col: 'close', open_col: 'open'})

test = test.drop(columns=[c for c in ['close', 'open'] if c in test.columns], errors='ignore')
test = test.merge(mkt, on='date', how='left').sort_values('date').reset_index(drop=True)
test['close_fwd21'] = test['close'].shift(-21)
test['ret_21d'] = np.log(test['close_fwd21'] / test['close'])
test['bh_ret_daily'] = np.log(test['close'] / test['close'].shift(1)).fillna(0)

signal = (test['regime_fixed'] == 'High-VIX') & (test['stack_pred'] == 1) & test['ret_21d'].notna() & test['open'].notna()
strat_daily = np.zeros(len(test))
in_position = np.zeros(len(test), dtype=int)
trade_windows = []
last_exit = -1
for i in range(len(test)):
    if not signal.iloc[i] or i <= last_exit:
        continue
    entry, exit_ = i + 1, i + 21
    if entry >= len(test) or exit_ >= len(test):
        continue
    if pd.isna(test.loc[entry, 'open']) or pd.isna(test.loc[exit_, 'close']):
        continue
    day1_ret = np.log(test.loc[entry, 'close'] / test.loc[entry, 'open'])
    strat_daily[entry] += day1_ret
    if exit_ > entry:
        strat_daily[entry + 1: exit_ + 1] += test.loc[entry + 1: exit_, 'bh_ret_daily'].values
    in_position[entry: exit_ + 1] = 1
    trade_windows.append((test.loc[entry, 'date'], test.loc[exit_, 'date']))
    last_exit = exit_

test['strat_ret_daily'] = strat_daily
test['in_position'] = in_position
test['strat_cum'] = test['strat_ret_daily'].cumsum()
test['bh_cum'] = test['bh_ret_daily'].cumsum()
test['strat_equity'] = np.exp(test['strat_cum'])
test['bh_equity'] = np.exp(test['bh_cum'])
test['strat_dd'] = test['strat_equity'] / test['strat_equity'].cummax() - 1
test['bh_dd'] = test['bh_equity'] / test['bh_equity'].cummax() - 1

print(f"Reconstructed trade windows: {trade_windows}")
print(f"Final strat equity: {test['strat_equity'].iloc[-1]:.4f}, BH equity: {test['bh_equity'].iloc[-1]:.4f}")
print(f"Strat max DD: {test['strat_dd'].min()*100:.2f}%, BH max DD: {test['bh_dd'].min()*100:.2f}%")


### Equity curve figure

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True, gridspec_kw={'height_ratios': [2.2, 1]})
ax1.plot(test['date'], test['bh_equity'], '--', color='grey', label='Buy-and-Hold')
ax1.plot(test['date'], test['strat_equity'], color='tomato', linewidth=1.8, label='High-VIX + Up (corrected: 1 trade, 21 active days)')
for (s, e) in trade_windows:
    ax1.axvspan(s, e, color='tomato', alpha=0.15)
ax1.set_ylabel('Portfolio Value (start = 1.0)')
ax1.set_title('Equity Curves — Bank Nifty Test Period (Jan 2025 – Feb 2026)\nCorrected protocol: signal at close[t], entry at open[t+1], exit at close[t+21], single position only')
ax1.legend(fontsize=8, loc='upper left')

ax2b = ax2.twinx()
ax2b.fill_between(test['date'], 0, test['india_vix'], color='salmon', alpha=0.4)
ax2b.axhline(18.71, color='darkred', linestyle='--', linewidth=1.2)
ax2b.text(test['date'].iloc[5], 18.71 + 0.5, 'VIX p75 threshold (18.71, fixed pre-test)', fontsize=8, color='darkred')
ax2b.set_ylabel('India VIX')
ax2.set_yticks([])
ax2.set_xlabel('Date')
plt.tight_layout()
plt.savefig(PLOTS / 'R12_Fig5_equity_curves.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved Fig 5')


### Drawdown figure

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(test['date'], test['bh_dd']*100, '--', color='grey', label='Buy-and-Hold')
ax.plot(test['date'], test['strat_dd']*100, color='tomato', label='High-VIX + Up')
ax.set_ylabel('Drawdown (%)')
ax.set_xlabel('Date')
ax.set_title(f"Drawdown Profiles (Max DD: Strategy {test['strat_dd'].min()*100:.2f}% vs Buy-and-Hold {test['bh_dd'].min()*100:.2f}%)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(PLOTS / 'R12_Fig6_drawdown.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved Fig 6')


## 4. India VIX vs GARCH(1,1) Regime Comparison

Reads the corrected comparison table produced by NB-R11.

In [ ]:
# ============================================================
# FIG 8: VIX vs GARCH regime comparison (corrected, from NB-R11 output)
# ============================================================
t10 = pd.read_csv(RESULTS / 'table10_garch_comparison_CORRECTED.csv')
fig, ax = plt.subplots(figsize=(7, 4.5))
labels = t10['Regime'].tolist()
accs = t10['Accuracy_pct'].tolist()
ns = t10['n'].tolist()
colors = ['#D6604D', '#F4A582', '#2166AC', '#92C5DE', '#777777']
bars = ax.bar(range(len(labels)), accs, color=colors)
for i, (a, n) in enumerate(zip(accs, ns)):
    if not pd.isna(a):
        ax.text(i, a + 1.5, f'{a:.1f}%\n(n={n})', ha='center', fontsize=8)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(['VIX\nHigh', 'VIX\nLow', 'GARCH\nHigh', 'GARCH\nLow', 'VIX∩GARCH\nHigh'], fontsize=8)
ax.axhline(50, color='grey', linestyle=':', linewidth=1)
ax.set_ylabel('Directional Accuracy (%)')
ax.set_title('India VIX vs GARCH(1,1) Regime Comparison (corrected, pre-test thresholds)\nNeither regime split is statistically significant (Fisher p=1.000 for both)')
ax.set_ylim(0, 115)
plt.tight_layout()
plt.savefig(PLOTS / 'R12_Fig8_vix_vs_garch.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved Fig 8')

print('\nAll figures regenerated in plots/. Fig 7 and Fig 9 already corrected: '
      'plots/R09_trading_simulation.png, plots/R04_statistical_tests.png')


---
## Summary

**Pipeline stage:** 12 of 13 (see `notebooks/README.md` for the full pipeline map).

**Prior notebook:** `NB-R11_garch_and_threshold_extras.ipynb`

**Artifacts produced by this notebook:**

- `plots/R12_Fig3_regime_accuracy_by_horizon.png`
- `plots/R12_Fig4_auc_by_horizon.png`
- `plots/R12_Fig5_equity_curves.png`
- `plots/R12_Fig6_drawdown.png`
- `plots/R12_Fig8_vix_vs_garch.png`

**Next notebook:** `NB-R13_train_1d_5d_models.ipynb`
